# 05 — Fonctionnalités spécifiques à Supabase

Supabase apporte des fonctionnalités au-delà d'une base PostgreSQL classique :
- **Row Level Security (RLS)** et rôles
- **API REST auto-générée** vs connexion SQL directe (quand utiliser laquelle)
- **Realtime** — écouter les changements en direct et exploiter immédiatement l'événement
- **Storage** — stockage de fichiers

Ce notebook explique chaque brique avec des exemples concrets.


## 1. Row Level Security (RLS) et rôles

RLS permet de définir, **au niveau de la base de données**, quelles lignes
un utilisateur a le droit de lire/modifier — indépendamment du code Python.

- La clé **`anon`** respecte les policies RLS (comme un utilisateur normal).
- La clé **`service_role`** contourne RLS (à réserver au backend, jamais côté client).

```sql
-- Exemple de policy SQL (à exécuter dans le SQL editor de Supabase) :
ALTER TABLE public.orders ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Les utilisateurs voient seulement leurs commandes"
ON public.orders
FOR SELECT
USING (auth.uid() = user_id);
```

Côté Python, il suffit d'utiliser la bonne clé (`SUPABASE_KEY` = anon key en
général, jamais la service_role key dans un notebook partagé publiquement).


## 2. API REST (PostgREST) vs connexion SQL directe

| | API REST (SDK `supabase-py`) | SQL direct (SQLAlchemy) |
|---|---|---|
| Respecte RLS automatiquement | ✅ Oui | ⚠️ Seulement si tu utilises un rôle contraint |
| Requêtes complexes (jointures avancées, CTE, fonctions SQL) | ❌ Limité | ✅ Complet |
| Simplicité pour du CRUD basique | ✅ Très simple | Un peu plus verbeux |
| Auth / Realtime / Storage intégrés | ✅ Oui | ❌ Non |

**Règle pratique :** utilise le SDK pour 90% des besoins (CRUD + auth + realtime),
et passe en SQL direct seulement quand la requête est trop complexe pour l'API REST.


In [ ]:
import sys
sys.path.append("..")
from db_connection.supabase_connector import get_supabase_client

client = get_supabase_client()


## 3. Realtime — écouter les changements en direct

Supabase peut notifier ton code Python en temps réel dès qu'une ligne est
insérée/modifiée/supprimée dans une table (via WebSocket).

C'est un exemple typique où "exploiter le résultat" veut dire **réagir à
un événement immédiatement** plutôt que d'interroger la base périodiquement.


In [ ]:
# Exemple (nécessite un contexte async ou un event loop actif) :
#
# def handle_change(payload):
#     print("Changement détecté :", payload)
#     # -> exploitation immédiate : ex. mettre à jour un dashboard,
#     #    envoyer une notification, déclencher un calcul...
#
# channel = client.channel("realtime:public:orders")
# channel.on_postgres_changes(
#     event="INSERT",
#     schema="public",
#     table="orders",
#     callback=handle_change,
# )
# channel.subscribe()
#
# Documentation complète : https://supabase.com/docs/guides/realtime
print("Voir le code commenté ci-dessus pour un exemple d'écoute Realtime.")


## 4. Supabase Storage — stockage de fichiers

Utile quand le résultat d'un traitement (export CSV, image générée, rapport PDF)
doit être stocké et rendu accessible via une URL.


In [ ]:
# Exemple : uploader un fichier généré par notre pipeline d'exploitation
#
# with open("../exports/ca_par_categorie.csv", "rb") as f:
#     client.storage.from_("rapports").upload("ca_par_categorie.csv", f)
#
# url = client.storage.from_("rapports").get_public_url("ca_par_categorie.csv")
# print("Fichier disponible ici :", url)

print("Exemple d'upload Storage (voir code commenté ci-dessus).")


## Suite

➡️ Notebook **`06_projet_complet_pipeline.ipynb`** pour un exemple de bout en bout
qui combine connexion, requêtage, exploitation ET action métier.
